# TODO 6.5：在 A100 40GB 补跑 FP16 / INT8 Benchmark

已有 AWQ/GPTQ 结果来自 `NVIDIA A100-SXM4-40GB`。本 Notebook 只补跑 FP16 和 SmoothQuant INT8，并复用已有六份 INT4 Benchmark，避免重新量化或重新运行 AWQ/GPTQ。

运行前请在 Colab 选择 A100 GPU。初始化单元会严格检查设备名称必须为 `NVIDIA A100-SXM4-40GB`；如果分配到 80GB 版本会立即停止。全部单元执行完成后，将更新的 `out/quantization_comparison` 和 `out/int8_fp8/benchmark_*.json` 拉回项目。

In [ ]:
%pip -q install -U "nvidia-modelopt[hf]" accelerate safetensors
print('安装完成后请重启 Colab 会话，再从下一单元继续。')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
INT8_MANIFEST = PROJECT_DIR / 'out/int8_fp8/smoothquant-int8-w8a8/quantized_model_manifest.json'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
BENCHMARK_CONFIG = PROJECT_DIR / 'configs/quantization_benchmark.json'
COMPARISON_CONFIG = PROJECT_DIR / 'configs/quantization_comparison.json'
RESULT_DIR = PROJECT_DIR / 'out/quantization_comparison'
INT8_RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
EXPECTED_GPU = 'NVIDIA A100-SXM4-40GB'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.data.finqa_assets import load_assets
from src.quantization.comparison import build_quantization_comparison, render_markdown, validate_quantization_comparison
from src.quantization.int8_fp8 import load_smoothquant_service

assert torch.cuda.is_available(), '请切换到 NVIDIA A100 GPU。'
actual_gpu = torch.cuda.get_device_name(0)
assert actual_gpu == EXPECTED_GPU, f'需要 {EXPECTED_GPU}，当前为 {actual_gpu}。请重新分配运行时。'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
INT8_RESULT_DIR.mkdir(parents=True, exist_ok=True)
assets = load_assets(DATASET_DIR)
benchmark_config = json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
prompt = InferenceRequest.from_finqa_row(assets.quality_dev[0], request_id='comparison-benchmark').render_prompt()
expected_case_count = (
    len(benchmark_config['matrix']['batch_sizes'])
    * len(benchmark_config['matrix']['input_lengths'])
    * len(benchmark_config['matrix']['output_lengths'])
)
assert expected_case_count == 16, f'预期 16 个测试点，实际为 {expected_case_count}'
print('GPU:', actual_gpu)
print('测试点:', expected_case_count)

In [ ]:
int4_reports = sorted((PROJECT_DIR / 'out/awq').glob('*/comparison_benchmark_report.json'))
int4_reports += sorted((PROJECT_DIR / 'out/gptq').glob('*/comparison_benchmark_report.json'))
assert len(int4_reports) == 6, f'应找到 6 份 AWQ/GPTQ 报告，实际找到 {len(int4_reports)} 份'
for report_path in int4_reports:
    existing_report = json.loads(report_path.read_text(encoding='utf-8'))
    report_gpu = existing_report['environment']['gpu']
    report_case_count = len(existing_report['cases'])
    assert report_gpu == EXPECTED_GPU, f'{report_path}: GPU 为 {report_gpu}'
    assert report_case_count == expected_case_count, f'{report_path}: 测试点为 {report_case_count}'
    print('可复用:', report_path.relative_to(PROJECT_DIR), report_gpu, report_case_count)

In [ ]:
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_a100_service.jsonl')
fp16_report = run_benchmark(fp16_service, prompt, benchmark_config)
fp16_path = RESULT_DIR / 'fp16_a100_benchmark_report.json'
fp16_path.write_text(json.dumps(fp16_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
fp16_validation = validate_benchmark_report(fp16_path)
(RESULT_DIR / 'fp16_a100_benchmark_validation.json').write_text(json.dumps(fp16_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert fp16_validation.complete
assert fp16_report['environment']['gpu'] == EXPECTED_GPU
del fp16_service
gc.collect()
torch.cuda.empty_cache()
print('FP16 A100 40GB Benchmark: PASS')

In [ ]:
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, INT8_RESULT_DIR / 'int8_benchmark_service.jsonl')
int8_report = run_benchmark(int8_service, prompt, benchmark_config)
int8_path = INT8_RESULT_DIR / 'benchmark_report.json'
int8_path.write_text(json.dumps(int8_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
int8_validation = validate_benchmark_report(int8_path)
(INT8_RESULT_DIR / 'benchmark_validation.json').write_text(json.dumps(int8_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert int8_validation.complete
assert int8_report['environment']['gpu'] == EXPECTED_GPU
del int8_service
gc.collect()
torch.cuda.empty_cache()
print('SmoothQuant INT8 A100 40GB Benchmark: PASS')

In [ ]:
report = build_quantization_comparison(PROJECT_DIR, COMPARISON_CONFIG)
report_path = RESULT_DIR / 'comparison_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(RESULT_DIR / 'comparison_report.md').write_text(render_markdown(report), encoding='utf-8')
validation = validate_quantization_comparison(report_path)
validation_dict = validation.to_dict()
(RESULT_DIR / 'validation.json').write_text(json.dumps(validation_dict, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
checks = {item['name']: item['status'] for item in validation_dict['checks']}
for check_name in ('memory_changes_recorded', 'latency_changes_recorded', 'throughput_changes_recorded'):
    assert checks[check_name] == 'pass', f'{check_name}: {checks[check_name]}'
print(json.dumps(validation_dict, ensure_ascii=False, indent=2))
print('性能三项已完成；若单位 token 成本仍为 not_run，只需补充 GPU 小时单价。')
print('请拉回:', RESULT_DIR)
print('请拉回:', int8_path)
print('请拉回:', INT8_RESULT_DIR / 'benchmark_validation.json')